In [2]:
# !pip install hyperopt

In [ ]:
# 하이퍼파라미터 튜닝: 모델의 최적의 성능을 낼 수 있는 Best Parameter(최적의 파라미터) 찾기.
# 하이퍼파라미터: 모델이 학습하는 방법 개발자가 값으로 지정하는 것. 이걸 모른다.
# GridSearchCV(그리드서치): 지정한 값들의 모든 조합을 전수조사, 시간이 많이 걸린다. 지정한 값이 베스트가 아니면 문제가 생긴다.
# RandomizedSearchCV(랜덤서치): 검색 공간에서 무작위로 n개 추출해서 best 찾기. (장)그리드서치보다 넓게 탐색 가능. (단)성능은 운이 좌우한다.
# HyperOpt: 지금까지의(입력, 결과) 쌍으로 대체 모델(surrogate model)을 만들고(학습하고),
#          가장 유망한 지정을 다음 후보로 선택해서 서치하는 방식.
#          (장) 적은 시도로 베스트 찾을 수 o(so, 시간 덜, 자원 덜). 연속 값을 가지는 파라미터에 특히 강함.
#          (단) 순차 실행. 병렬처리가 안 됨. 알고리즘 복잡해서 결과가 안 좋을 때가 있음.
#          (사용시점) 하이퍼파라미터 개수가 많고 복잡할 때.(성능 좋은 모델일수록 하이퍼파라미터가 많음. 성능 좋은 모델일 수록 hyperopt 쓰는 게 나음.)

# HyperOpt 4대 구성 요소
# 튜닝을 수행하려면 반드시 네 가지 준비
#   1. 검색 공간(Search Space): 각 하이퍼 파라미터의 값이 어떤 범위에서 어떤 분포로 추출(샘플링)될지를 정의
#   2. 목적 함수(Objective Function): 검색 공간에서 값을 하나 뽑아 "이 값이 얼마나 나쁜가(클수록 나쁨.)"를 실수로 반환.
#                                    HyperOpt는 이 반환 값을 최소화함(최소값을 찾음) => fmin() 사용.
#   3. 최적화 알고리즘(tpe.suggest): 다음에 시도할 후보를 어떻게 고를지 결정하는 것.(best parameter 있는 위치 찾는 것?)
#   4. 결과 저장 객체(Trials): 매 시도의 입력값(하이퍼파라미터의 값)과 반환값(목적함수)을 모두 기록하는 객체.(상태 기억)

In [ ]:
import hyperopt


print(hyperopt.__version__)
# 버전에 대한 주의 사항: hyperopt 0.2.5 이하 버전은 numpy 2.x에서 error
# 해결방법: 0.2.7 버전 이상으로 작업 진행 필요.

0.3.0


In [3]:
# 방정식 : f = x**2 - 20*y
from hyperopt import hp


# -10 ~ 10까지 1간격을 가지는 입력 변수 x와 -15 ~ 15까지 1간격으로 입력 변수 y 설정.
search_space = {'x': hp.quniform('x', -10, 10, 1), 'y': hp.quniform('y', -15, 15, 1) }


In [ ]:
from hyperopt import STATUS_OK


# 목적 함수를 생성. 변숫값과 변수 검색 공간을 가지는 딕셔너리를 인자로 받고, 특정 값을 반환
def objective_func(search_space):
    x = search_space['x']
    y = search_space['y']
    retval = x**2 - 20*y
   
    return retval
 

In [6]:
from hyperopt import fmin, tpe, Trials
import numpy as np


# 입력 결괏값을 저장한 Trials 객체값 생성.
trial_val = Trials()


# 목적 함수의 최솟값을 반환하는 최적 입력 변숫값을 5번의 입력값 시도(max_evals=5)로 찾아냄.
best_01 = fmin(fn=objective_func, space=search_space, algo=tpe.suggest, max_evals=5
               , trials=trial_val, rstate=np.random.default_rng(seed=0))
print('best:', best_01)


100%|██████████| 5/5 [00:00<00:00, 1001.55trial/s, best loss: -224.0]
best: {'x': np.float64(-4.0), 'y': np.float64(12.0)}


In [ ]:
trial_val = Trials()


# max_evals를 200회로 늘려서 재테스트
best_02 = fmin(fn=objective_func, space=search_space, algo=tpe.suggest, max_evals=200
               , trials=trial_val, rstate=np.random.default_rng(seed=0))
print('best:', best_02)
 

100%|██████████| 200/200 [00:01<00:00, 134.96trial/s, best loss: -300.0]
best: {'x': np.float64(0.0), 'y': np.float64(15.0)}


In [ ]:
# fmin( )에 인자로 들어가는 Trials 객체의 result 속성에 파이썬 리스트로 목적 함수 반환값들이 저장됨
# 리스트 내부의 개별 원소는 {'loss':함수 반환값, 'status':반환 상태값} 와 같은 딕셔너리임.
print(trial_val.results)

[{'loss': -64.0, 'status': 'ok'}, {'loss': -184.0, 'status': 'ok'}, {'loss': 56.0, 'status': 'ok'}, {'loss': -224.0, 'status': 'ok'}, {'loss': 61.0, 'status': 'ok'}, {'loss': -296.0, 'status': 'ok'}, {'loss': -40.0, 'status': 'ok'}, {'loss': 281.0, 'status': 'ok'}, {'loss': 64.0, 'status': 'ok'}, {'loss': 100.0, 'status': 'ok'}, {'loss': 60.0, 'status': 'ok'}, {'loss': -39.0, 'status': 'ok'}, {'loss': 1.0, 'status': 'ok'}, {'loss': -164.0, 'status': 'ok'}, {'loss': 21.0, 'status': 'ok'}, {'loss': -56.0, 'status': 'ok'}, {'loss': 284.0, 'status': 'ok'}, {'loss': 176.0, 'status': 'ok'}, {'loss': -171.0, 'status': 'ok'}, {'loss': 0.0, 'status': 'ok'}, {'loss': -291.0, 'status': 'ok'}, {'loss': -271.0, 'status': 'ok'}, {'loss': -255.0, 'status': 'ok'}, {'loss': -156.0, 'status': 'ok'}, {'loss': -291.0, 'status': 'ok'}, {'loss': -136.0, 'status': 'ok'}, {'loss': -211.0, 'status': 'ok'}, {'loss': -176.0, 'status': 'ok'}, {'loss': -95.0, 'status': 'ok'}, {'loss': -264.0, 'status': 'ok'}, {'lo

In [ ]:
import pandas as pd


# results에서 loss 키값에 해당하는 밸류들을 추출하여 list로 생성.
losses = [loss_dict['loss'] for loss_dict in trial_val.results]


# DataFrame으로 생성.
result_df = pd.DataFrame({'x': trial_val.vals['x'], 'y': trial_val.vals['y'], 'losses': losses})
result_df

,x,y,losses
0,-6.0,5.0,-64.0
1,-4.0,10.0,-184.0
2,4.0,-2.0,56.0
3,-4.0,12.0,-224.0
4,9.0,1.0,61.0
...,...,...,...
195,-2.0,11.0,-216.0
196,-3.0,5.0,-91.0
197,0.0,7.0,-140.0
198,-1.0,2.0,-39.0


In [14]:
# HyperOpt를 이용한 XGBoost 하이퍼 파라미터 최적화
# 아래 코드는 이전에 수록된 코드라 책에는 싣지 않았습니다.
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')


dataset = load_breast_cancer()


cancer_df = pd.DataFrame(data=dataset.data, columns=dataset.feature_names)
cancer_df['target']= dataset.target
X_features = cancer_df.iloc[:, :-1]
y_label = cancer_df.iloc[:, -1]


In [15]:
# 전체 데이터 중 80%는 학습용 데이터, 20%는 테스트용 데이터 추출
X_train, X_test, y_train, y_test=train_test_split(X_features, y_label, test_size=0.2, random_state=156 )


# 앞에서 추출한 학습 데이터를 다시 학습과 검증 데이터로 분리
X_tr, X_val, y_tr, y_val= train_test_split(X_train, y_train, test_size=0.1, random_state=156 )


In [16]:
from hyperopt import hp


# max_depth는 5에서 20까지 1간격으로, min_child_weight는 1에서 2까지 1간격으로
# colsample_bytree는 0.5에서 1사이, learning_rate는 0.01에서 0.2 사이 정규 분포된 값으로 검색.
xgb_search_space = {'max_depth': hp.quniform('max_depth', 5, 20, 1),
                    'min_child_weight': hp.quniform('min_child_weight', 1, 2, 1),
                    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),
                    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1),
                   }


In [17]:
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
from hyperopt import STATUS_OK


# fmin()에서 입력된 search_space 값으로 입력된 모든 값은 실수형임.
# XGBClassifier의 정수형 하이퍼 파라미터는 정수형 변환을 해줘야 함.
# 정확도는 높을수록 더 좋은 수치임. -1 * 정확도를 곱해서 큰 정확도 값일수록 최소가 되도록 변환
def objective_func(search_space):
    # 수행 시간 절약을 위해 nestimators는 100으로 축소
    xgb_clf = XGBClassifier(n_estimators=100, max_depth=int(search_space['max_depth']),
                            min_child_weight=int(search_space['min_child_weight']),
                            learning_rate=search_space['learning_rate'],
                            colsample_bytree=search_space['colsample_bytree'],
                            eval_metric='logloss')
    accuracy = cross_val_score(xgb_clf, X_train, y_train, scoring='accuracy', cv=3)
   
    # accuracy는 cv=3 개수만큼 roc-auc 결과를 리스트로 가짐. 이를 평균해서 반환하되 -1을 곱함.
    return {'loss':-1 * np.mean(accuracy), 'status': STATUS_OK}




In [ ]:
from hyperopt import fmin, tpe, Trials


trial_val = Trials()
best = fmin(fn=objective_func,
            space=xgb_search_space,
            algo=tpe.suggest,
            max_evals=100, # 최대 반복 횟수를 지정합니다.
            trials=trial_val, rstate=np.random.default_rng(seed=9))
print('best:', best)


100%|██████████| 100/100 [00:19<00:00,  5.13trial/s, best loss: -0.967047170907401]
best: {'colsample_bytree': np.float64(0.7923560109541058), 'learning_rate': np.float64(0.1938940946651398), 'max_depth': np.float64(16.0), 'min_child_weight': np.float64(2.0)}


In [24]:
print('colsample_bytree:{0}, learning_rate:{1}, max_depth:{2}, min_child_weight:{3}'.format(
    round(best['colsample_bytree'], 5), round(best['learning_rate'], 5),
    int(best['max_depth']), int(best['min_child_weight'])))


colsample_bytree:0.79236, learning_rate:0.19389, max_depth:16, min_child_weight:2


In [25]:
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import f1_score, roc_auc_score


def get_clf_eval(y_test, pred=None, pred_proba=None):
    confusion = confusion_matrix( y_test, pred)
    accuracy = accuracy_score(y_test , pred)
    precision = precision_score(y_test , pred)
    recall = recall_score(y_test , pred)
    f1 = f1_score(y_test,pred)
    # ROC-AUC 추가
    roc_auc = roc_auc_score(y_test, pred_proba)
    print('오차 행렬')
    print(confusion)
    # ROC-AUC print 추가
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f},\
    F1: {3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))


In [26]:
xgb_wrapper = XGBClassifier(n_estimators=400,
                            learning_rate=round(best['learning_rate'], 5),
                            max_depth=int(best['max_depth']),
                            min_child_weight=int(best['min_child_weight']),
                            colsample_bytree=round(best['colsample_bytree'], 5),
                            early_stopping_rounds=50,
                             eval_metric='logloss'
                           )


evals = [(X_tr, y_tr), (X_val, y_val)]
xgb_wrapper.fit(X_tr, y_tr, 
                eval_set=evals, verbose=True)


preds = xgb_wrapper.predict(X_test)
pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]


get_clf_eval(y_test, preds, pred_proba)


[0]	validation_0-logloss:0.52017	validation_1-logloss:0.56664
[1]	validation_0-logloss:0.41762	validation_1-logloss:0.49501
[2]	validation_0-logloss:0.34123	validation_1-logloss:0.43295
[3]	validation_0-logloss:0.28440	validation_1-logloss:0.39091
[4]	validation_0-logloss:0.24056	validation_1-logloss:0.35565
[5]	validation_0-logloss:0.20676	validation_1-logloss:0.33910
[6]	validation_0-logloss:0.17704	validation_1-logloss:0.31555
[7]	validation_0-logloss:0.15563	validation_1-logloss:0.29982
[8]	validation_0-logloss:0.13628	validation_1-logloss:0.28915
[9]	validation_0-logloss:0.11984	validation_1-logloss:0.27626
[10]	validation_0-logloss:0.10728	validation_1-logloss:0.27569
[11]	validation_0-logloss:0.09639	validation_1-logloss:0.27079
[12]	validation_0-logloss:0.08691	validation_1-logloss:0.26944
[13]	validation_0-logloss:0.07998	validation_1-logloss:0.26511


[14]	validation_0-logloss:0.07359	validation_1-logloss:0.26765
[15]	validation_0-logloss:0.06817	validation_1-logloss:0.26790
[16]	validation_0-logloss:0.06358	validation_1-logloss:0.26820
[17]	validation_0-logloss:0.05831	validation_1-logloss:0.26605
[18]	validation_0-logloss:0.05449	validation_1-logloss:0.26464
[19]	validation_0-logloss:0.05087	validation_1-logloss:0.26190
[20]	validation_0-logloss:0.04734	validation_1-logloss:0.26330
[21]	validation_0-logloss:0.04406	validation_1-logloss:0.25782
[22]	validation_0-logloss:0.04097	validation_1-logloss:0.25848
[23]	validation_0-logloss:0.03889	validation_1-logloss:0.25730
[24]	validation_0-logloss:0.03674	validation_1-logloss:0.25660
[25]	validation_0-logloss:0.03510	validation_1-logloss:0.25804
[26]	validation_0-logloss:0.03349	validation_1-logloss:0.25645
[27]	validation_0-logloss:0.03152	validation_1-logloss:0.25428
[28]	validation_0-logloss:0.03017	validation_1-logloss:0.25443
[29]	validation_0-logloss:0.02943	validation_1-logloss: